Q1

In [2]:
import nltk
import pandas as pd
import numpy as np
import re
import math
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents_q1 = [
    "Machine learning is useful for data analysis. Machine learning finds patterns in data.",
    "Deep learning uses neural networks for image and text analysis. Neural networks learn complex patterns.",
    "Natural language processing helps computers understand human language. Language models process text data.",
    "Data science combines statistics, programming, and machine learning. Data analysis supports business decisions.",
    "Computer vision analyzes images using deep learning and neural networks. Image analysis is useful in healthcare."
]

def preprocess(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return [word for word in tokens if word not in ENGLISH_STOP_WORDS]

processed_q1 = [preprocess(doc) for doc in documents_q1]

print("Preprocessed Corpus:")
for i, tokens in enumerate(processed_q1, 1):
    print(f"Doc{i}: {tokens}")


Preprocessed Corpus:
Doc1: ['machine', 'learning', 'useful', 'data', 'analysis', 'machine', 'learning', 'finds', 'patterns', 'data']
Doc2: ['deep', 'learning', 'uses', 'neural', 'networks', 'image', 'text', 'analysis', 'neural', 'networks', 'learn', 'complex', 'patterns']
Doc3: ['natural', 'language', 'processing', 'helps', 'computers', 'understand', 'human', 'language', 'language', 'models', 'process', 'text', 'data']
Doc4: ['data', 'science', 'combines', 'statistics', 'programming', 'machine', 'learning', 'data', 'analysis', 'supports', 'business', 'decisions']
Doc5: ['computer', 'vision', 'analyzes', 'images', 'using', 'deep', 'learning', 'neural', 'networks', 'image', 'analysis', 'useful', 'healthcare']


In [3]:
vocabulary_q1 = sorted(set(token for doc in processed_q1 for token in doc))
print("Vocabulary:", vocabulary_q1)
print("Vocabulary size:", len(vocabulary_q1))

tf_matrix = []
for doc in processed_q1:
    counts = Counter(doc)
    tf_matrix.append([counts[term] for term in vocabulary_q1])

tf_df = pd.DataFrame(tf_matrix, index=[f"Doc{i}" for i in range(1, 6)], columns=vocabulary_q1)
print("\nTF Matrix:")
print(tf_df)


Vocabulary: ['analysis', 'analyzes', 'business', 'combines', 'complex', 'computer', 'computers', 'data', 'decisions', 'deep', 'finds', 'healthcare', 'helps', 'human', 'image', 'images', 'language', 'learn', 'learning', 'machine', 'models', 'natural', 'networks', 'neural', 'patterns', 'process', 'processing', 'programming', 'science', 'statistics', 'supports', 'text', 'understand', 'useful', 'uses', 'using', 'vision']
Vocabulary size: 37

TF Matrix:
      analysis  analyzes  business  combines  complex  computer  computers  \
Doc1         1         0         0         0        0         0          0   
Doc2         1         0         0         0        1         0          0   
Doc3         0         0         0         0        0         0          1   
Doc4         1         0         1         1        0         0          0   
Doc5         1         1         0         0        0         1          0   

      data  decisions  deep  ...  programming  science  statistics  supports  

In [4]:
N = len(processed_q1)
idf_values = {}

for term in vocabulary_q1:
    df = sum(term in doc for doc in processed_q1)
    idf_values[term] = math.log(N / df)

idf_df = pd.DataFrame({"IDF": idf_values})
print("IDF Values:")
print(idf_df)


IDF Values:
                  IDF
analysis     0.223144
analyzes     1.609438
business     1.609438
combines     1.609438
complex      1.609438
computer     1.609438
computers    1.609438
data         0.510826
decisions    1.609438
deep         0.916291
finds        1.609438
healthcare   1.609438
helps        1.609438
human        1.609438
image        0.916291
images       1.609438
language     1.609438
learn        1.609438
learning     0.223144
machine      0.916291
models       1.609438
natural      1.609438
networks     0.916291
neural       0.916291
patterns     0.916291
process      1.609438
processing   1.609438
programming  1.609438
science      1.609438
statistics   1.609438
supports     1.609438
text         0.916291
understand   1.609438
useful       0.916291
uses         1.609438
using        1.609438
vision       1.609438


In [5]:
tfidf_manual = tf_df.astype(float).copy()

for term in vocabulary_q1:
    tfidf_manual[term] = tf_df[term] * idf_values[term]

print("TF-IDF Matrix:")
print(tfidf_manual.round(4))


TF-IDF Matrix:
      analysis  analyzes  business  combines  complex  computer  computers  \
Doc1    0.2231    0.0000    0.0000    0.0000   0.0000    0.0000     0.0000   
Doc2    0.2231    0.0000    0.0000    0.0000   1.6094    0.0000     0.0000   
Doc3    0.0000    0.0000    0.0000    0.0000   0.0000    0.0000     1.6094   
Doc4    0.2231    0.0000    1.6094    1.6094   0.0000    0.0000     0.0000   
Doc5    0.2231    1.6094    0.0000    0.0000   0.0000    1.6094     0.0000   

        data  decisions    deep  ...  programming  science  statistics  \
Doc1  1.0217     0.0000  0.0000  ...       0.0000   0.0000      0.0000   
Doc2  0.0000     0.0000  0.9163  ...       0.0000   0.0000      0.0000   
Doc3  0.5108     0.0000  0.0000  ...       0.0000   0.0000      0.0000   
Doc4  1.0217     1.6094  0.0000  ...       1.6094   1.6094      1.6094   
Doc5  0.0000     0.0000  0.9163  ...       0.0000   0.0000      0.0000   

      supports    text  understand  useful    uses   using  vision  
Do

In [6]:
print("Most important terms in each document:")
for i in range(len(documents_q1)):
    important = tfidf_manual.iloc[i].sort_values(ascending=False).head(5)
    print(f"\nDoc{i+1}:")
    print(important.round(4))


Most important terms in each document:

Doc1:
machine     1.8326
finds       1.6094
data        1.0217
patterns    0.9163
useful      0.9163
Name: Doc1, dtype: float64

Doc2:
neural      1.8326
networks    1.8326
learn       1.6094
uses        1.6094
complex     1.6094
Name: Doc2, dtype: float64

Doc3:
language     4.8283
helps        1.6094
human        1.6094
computers    1.6094
models       1.6094
Name: Doc3, dtype: float64

Doc4:
business     1.6094
combines     1.6094
science      1.6094
supports     1.6094
decisions    1.6094
Name: Doc4, dtype: float64

Doc5:
analyzes      1.6094
vision        1.6094
images        1.6094
healthcare    1.6094
computer      1.6094
Name: Doc5, dtype: float64


Q2

In [7]:
documents_q2 = [
    "The football team won the match after scoring three goals.",
    "The football players trained hard before the important match.",
    "The coach prepared the team for the football tournament.",
    "The basketball team won the game with excellent defense.",
    "Basketball players practiced shooting before the game.",
    "The coach planned basketball training for the tournament.",
    "The new smartphone has a powerful processor and camera.",
    "Technology companies released a new smartphone with better battery life.",
    "The laptop has a fast processor and large storage.",
    "Students use laptops and smartphones for online learning."
]

vectorizer = TfidfVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]+\b")
document_term_matrix = vectorizer.fit_transform(documents_q2)

print("Document-term matrix shape:", document_term_matrix.shape)
print("Dimensions = documents × terms")


Document-term matrix shape: (10, 42)
Dimensions = documents × terms


In [8]:
feature_names = vectorizer.get_feature_names_out()
dtm_df = pd.DataFrame(
    document_term_matrix.toarray(),
    index=[f"Doc{i}" for i in range(1, 11)],
    columns=feature_names
)

print("Document-Term Matrix:")
print(dtm_df.round(4))


Document-Term Matrix:
       basketball  battery  better  camera   coach  companies  defense  \
Doc1       0.0000   0.0000  0.0000  0.0000  0.0000     0.0000   0.0000   
Doc2       0.0000   0.0000  0.0000  0.0000  0.0000     0.0000   0.0000   
Doc3       0.0000   0.0000  0.0000  0.0000  0.4511     0.0000   0.0000   
Doc4       0.3486   0.0000  0.0000  0.0000  0.0000     0.0000   0.4687   
Doc5       0.3719   0.0000  0.0000  0.0000  0.0000     0.0000   0.0000   
Doc6       0.3719   0.0000  0.0000  0.0000  0.4251     0.0000   0.0000   
Doc7       0.0000   0.0000  0.0000  0.4898  0.0000     0.0000   0.0000   
Doc8       0.0000   0.3665  0.3665  0.0000  0.0000     0.3665   0.0000   
Doc9       0.0000   0.0000  0.0000  0.0000  0.0000     0.0000   0.0000   
Doc10      0.0000   0.0000  0.0000  0.0000  0.0000     0.0000   0.0000   

       excellent    fast  football  ...  smartphones  storage  students  \
Doc1      0.0000  0.0000    0.3486  ...       0.0000   0.0000    0.0000   
Doc2      0.0

In [9]:
document_vectors = document_term_matrix.toarray()

print("First three document vectors:")
for i in range(3):
    print(f"Doc{i+1}:", np.round(document_vectors[i], 4))


First three document vectors:
Doc1: [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.3486
 0.     0.4687 0.     0.     0.     0.     0.     0.     0.     0.3985
 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.4687
 0.     0.     0.     0.     0.     0.3486 0.     0.     0.     0.
 0.     0.3985]
Doc2: [0.     0.     0.     0.     0.     0.     0.     0.     0.     0.3327
 0.     0.     0.4473 0.4473 0.     0.     0.     0.     0.     0.3802
 0.     0.     0.     0.3802 0.     0.     0.     0.     0.     0.
 0.     0.     0.     0.     0.     0.     0.     0.     0.4473 0.
 0.     0.    ]
Doc3: [0.     0.     0.     0.     0.4511 0.     0.     0.     0.     0.3946
 0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
 0.     0.     0.     0.     0.     0.     0.5306 0.     0.     0.
 0.     0.     0.     0.     0.     0.3946 0.     0.4511 0.     0.
 0.     0.    ]


In [10]:
pairs_q2 = [(1, 2), (1, 4), (7, 8)]

print("Cosine Similarity:")
for a, b in pairs_q2:
    score = cosine_similarity(document_term_matrix[a-1], document_term_matrix[b-1])[0][0]
    print(f"Doc{a} and Doc{b}: {score:.4f}")


Cosine Similarity:
Doc1 and Doc2: 0.2675
Doc1 and Doc4: 0.2803
Doc7 and Doc8: 0.2595


**Observation:** In the Vector Space Model, every document becomes a numerical vector. Documents with more similar word distributions generally have higher cosine similarity.

## Q3. Distributional Representation and Word Representation

In [11]:
corpus_q3 = [
    "Machine learning algorithms learn patterns from data.",
    "Machine learning models analyze data for prediction.",
    "Deep learning models use neural networks for prediction.",
    "Neural networks learn useful representations from data.",
    "Data science uses statistics and machine learning.",
    "Data analysis helps scientists understand patterns.",
    "Natural language processing analyzes human language.",
    "Language models learn patterns from text data.",
    "Text processing is important for natural language applications.",
    "Computer vision analyzes images using deep learning.",
    "Image recognition uses neural networks and machine learning.",
    "Deep learning improves image classification.",
    "Students study machine learning and data science.",
    "Students learn programming and statistics.",
    "Programming skills help students build machine learning models.",
    "Researchers analyze data to discover useful patterns.",
    "Researchers use neural networks for image analysis.",
    "Scientists use statistics to analyze experimental data.",
    "Machine learning is used in healthcare and finance.",
    "Healthcare systems use data analysis for prediction.",
    "Finance applications use machine learning for forecasting.",
    "Text classification uses language features and machine learning.",
    "Language processing helps computers understand text.",
    "Computers process language using statistical models.",
    "Models learn from training data.",
    "Training data contains examples for learning.",
    "Good data improves machine learning performance.",
    "Neural models learn representations from examples.",
    "Deep models can learn complex patterns.",
    "Learning algorithms require useful training data.",
    "Data preprocessing improves model performance.",
    "Text preprocessing removes unnecessary words.",
    "Image preprocessing improves computer vision systems.",
    "Statistics helps evaluate machine learning models.",
    "Prediction models are tested using data.",
    "Machine learning can support scientific research.",
    "Natural language applications process large text collections.",
    "Computer vision applications process image collections.",
    "Researchers compare models using evaluation measures.",
    "Students practice machine learning using real datasets."
]

processed_q3 = [preprocess(s) for s in corpus_q3]
all_words = [w for s in processed_q3 for w in s]
word_counts = Counter(all_words)
selected_words = [w for w, _ in word_counts.most_common(10)]

print("Number of sentences:", len(corpus_q3))
print("Ten frequent words:", selected_words)


Number of sentences: 40
Ten frequent words: ['learning', 'data', 'machine', 'models', 'language', 'learn', 'text', 'patterns', 'use', 'neural']


In [12]:
# Co-occurrence matrix with a one-word context window.
cooccurrence = pd.DataFrame(0, index=selected_words, columns=selected_words, dtype=int)

for sentence in processed_q3:
    for i, word in enumerate(sentence):
        if word not in selected_words:
            continue
        for j in range(max(0, i-1), min(len(sentence), i+2)):
            if i != j and sentence[j] in selected_words:
                cooccurrence.loc[word, sentence[j]] += 1

print("Word-context / co-occurrence matrix:")
print(cooccurrence)
print("\nWord-vector dimensions:", cooccurrence.shape)


Word-context / co-occurrence matrix:
          learning  data  machine  models  language  learn  text  patterns  \
learning         0     1       13       4         0      0     0         0   
data             1     0        0       0         0      0     1         1   
machine         13     0        0       0         0      0     0         0   
models           4     0        0       0         1      4     0         0   
language         0     0        0       1         0      0     0         0   
learn            0     0        0       4         0      0     0         2   
text             0     1        0       0         0      0     0         1   
patterns         0     1        0       0         0      2     1         0   
use              0     1        1       1         0      0     0         0   
neural           0     0        0       1         0      0     0         0   

          use  neural  
learning    0       0  
data        1       0  
machine     1       0  
models  

In [13]:
print("Selected word vectors:")
for word in selected_words:
    print(f"{word}: {cooccurrence.loc[word].values}")


Selected word vectors:
learning: [ 0  1 13  4  0  0  0  0  0  0]
data: [1 0 0 0 0 0 1 1 1 0]
machine: [13  0  0  0  0  0  0  0  1  0]
models: [4 0 0 0 1 4 0 0 1 1]
language: [0 0 0 1 0 0 0 0 0 0]
learn: [0 0 0 4 0 0 0 2 0 0]
text: [0 1 0 0 0 0 0 1 0 0]
patterns: [0 1 0 0 0 2 1 0 0 0]
use: [0 1 1 1 0 0 0 0 0 2]
neural: [0 0 0 1 0 0 0 0 2 0]


In [14]:
word_pairs = [
    (selected_words[0], selected_words[1]),
    (selected_words[0], selected_words[2]),
    (selected_words[1], selected_words[2])
]

print("Word-vector Cosine Similarity:")
for a, b in word_pairs:
    score = cosine_similarity(
        cooccurrence.loc[a].values.reshape(1, -1),
        cooccurrence.loc[b].values.reshape(1, -1)
    )[0][0]
    print(f"{a} - {b}: {score:.4f}")


Word-vector Cosine Similarity:
learning - data: 0.0000
learning - machine: 0.0000
data - machine: 0.5369


**Observation:** The distributional hypothesis says that words appearing in similar contexts tend to have similar meanings or uses. The co-occurrence vector represents a word using its nearby context words.

## Q4. Sentence Representation and Document-to-Vector Representation

In [15]:
sentences_q4 = [
    "Machine learning models analyze data.",
    "Deep learning models learn complex patterns from data.",
    "Students study machine learning in class.",
    "Data science uses statistics and machine learning.",
    "Football players train hard for matches.",
    "The football team won the important match.",
    "Basketball players practice before the game.",
    "The basketball team prepared for the tournament.",
    "Natural language processing analyzes text.",
    "Language models learn patterns from text."
]

sentence_vectorizer = TfidfVectorizer(stop_words="english", token_pattern=r"(?u)\b[a-zA-Z]+\b")
sentence_matrix = sentence_vectorizer.fit_transform(sentences_q4)
sentence_vectors = sentence_matrix.toarray()

print("Number of sentences:", len(sentences_q4))
print("Sentence-vector matrix shape:", sentence_matrix.shape)


Number of sentences: 10
Sentence-vector matrix shape: (10, 34)


In [16]:
sentence_df = pd.DataFrame(
    sentence_vectors,
    index=[f"Sentence{i}" for i in range(1, 11)],
    columns=sentence_vectorizer.get_feature_names_out()
)

print("Sentence vectors:")
print(sentence_df.round(4))


Sentence vectors:
            analyze  analyzes  basketball   class  complex    data    deep  \
Sentence1    0.5683    0.0000       0.000  0.0000   0.0000  0.4226  0.0000   
Sentence2    0.0000    0.0000       0.000  0.0000   0.4477  0.3330  0.4477   
Sentence3    0.0000    0.0000       0.000  0.5006   0.0000  0.0000  0.0000   
Sentence4    0.0000    0.0000       0.000  0.0000   0.0000  0.3489  0.0000   
Sentence5    0.0000    0.0000       0.000  0.0000   0.0000  0.0000  0.0000   
Sentence6    0.0000    0.0000       0.000  0.0000   0.0000  0.0000  0.0000   
Sentence7    0.0000    0.0000       0.458  0.0000   0.0000  0.0000  0.0000   
Sentence8    0.0000    0.0000       0.458  0.0000   0.0000  0.0000  0.0000   
Sentence9    0.0000    0.4743       0.000  0.0000   0.0000  0.0000  0.0000   
Sentence10   0.0000    0.0000       0.000  0.0000   0.0000  0.0000  0.0000   

            football    game    hard  ...  science  statistics  students  \
Sentence1     0.0000  0.0000  0.0000  ...   0.0

In [17]:
sentence_pairs = [(1, 2), (1, 4), (5, 6), (7, 8), (9, 10)]

print("Sentence Cosine Similarity:")
for a, b in sentence_pairs:
    score = cosine_similarity(sentence_matrix[a-1], sentence_matrix[b-1])[0][0]
    print(f"Sentence{a} and Sentence{b}: {score:.4f}")


Sentence Cosine Similarity:
Sentence1 and Sentence2: 0.3927
Sentence1 and Sentence4: 0.4115
Sentence5 and Sentence6: 0.1626
Sentence7 and Sentence8: 0.2098
Sentence9 and Sentence10: 0.3694


In [18]:
# Find the most similar pairs among all ten sentences.
similarities = []

for i in range(len(sentences_q4)):
    for j in range(i + 1, len(sentences_q4)):
        score = cosine_similarity(sentence_matrix[i], sentence_matrix[j])[0][0]
        similarities.append((score, i+1, j+1))

similarities.sort(reverse=True)

print("Most similar sentence pairs:")
for score, a, b in similarities[:5]:
    print(f"Sentence{a} - Sentence{b}: {score:.4f}")


Most similar sentence pairs:
Sentence2 - Sentence10: 0.4821
Sentence1 - Sentence4: 0.4115
Sentence1 - Sentence2: 0.3927
Sentence9 - Sentence10: 0.3694
Sentence1 - Sentence3: 0.2817


In [19]:
# Document-to-vector representation: mean of all sentence vectors.
document_vector = np.mean(sentence_vectors, axis=0)

print("Document vector:")
print(np.round(document_vector, 4))
print("Document vector dimension:", document_vector.shape[0])


Document vector:
[0.0568 0.0474 0.0916 0.0501 0.0448 0.1105 0.0448 0.0806 0.0539 0.0474
 0.0474 0.0861 0.0839 0.1313 0.1144 0.0474 0.0474 0.1156 0.0474 0.0839
 0.0861 0.0539 0.0539 0.0474 0.0469 0.0469 0.0501 0.0501 0.0861 0.0861
 0.0539 0.0474 0.0469 0.0474]
Document vector dimension: 34


### Comparison of representations

- **Word representation:** represents one word using its context. In Q3, each selected word has a vector with 10 dimensions.
- **Sentence representation:** represents one sentence using TF-IDF values. Its dimension equals the sentence vocabulary size.
- **Document representation:** combines multiple sentence vectors into one fixed-length vector.

A fixed-length representation can lose word order, detailed context, and some relationships between words when a complete sentence or document is compressed into one vector.